In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import wandb
# 设置中文字体（以黑体为例）
plt.rcParams['font.sans-serif'] = ['SimHei'] 
# 解决负号 '-' 显示为方块的问题
plt.rcParams['axes.unicode_minus'] = False
# 1. 初始化 WandB 项目
wandb.init(project="power-load-analysis", name="load-visualization-run")

def visualize_power_data(file_path):
    # 2. 读取数据
    # 假设你的列名分别是 '数据时间' 和 '总有功功率（kw）'
    df = pd.read_csv(file_path)
    
    # 转换为 datetime 格式以便正确绘图
    df['数据时间'] = pd.to_datetime(df['数据时间'])
    
    # 3. 使用 Matplotlib 绘制总体趋势图
    plt.figure(figsize=(15, 6))
    plt.plot(df['数据时间'], df['总有功功率（kw）'], color='#1f77b4', linewidth=0.5)
    plt.title('某地区电力负荷年度总览')
    plt.xlabel('时间')
    plt.ylabel('功率 (kW)')
    plt.grid(True, alpha=0.3)
    
    # 4. 将 Matplotlib 图像保存并记录到 WandB
    wandb.log({"Power_Load_Overview_Plot": wandb.Image(plt)})
    plt.close()

    # 5. (进阶) 上传交互式图表：将数据存入 WandB Table
    # 为了防止数据量过大导致网页卡顿，可以对展示数据进行适当采样（例如每4个点取1个）
    sample_df = df.iloc[::4, :] 
    
    table = wandb.Table(dataframe=sample_df)
    wandb.log({
        "Interactive_Load_Chart": wandb.plot.line(
            table, "数据时间", "总有功功率（kw）", 
            title="电力负荷交互式曲线"
        )
    })

    print("数据已成功上传至 WandB 仪表盘！")

# 调用函数（替换为你的文件名）
visualize_power_data('load.csv')

# 结束 WandB 运行
wandb.finish()

数据已成功上传至 WandB 仪表盘！


In [27]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from pandas import DataFrame
from sklearn.preprocessing import StandardScaler
import pandas as pd
file_path = 'load.csv'
data = pd.read_csv('load.csv', header=None, skiprows=1)
# print(data[:10])
power_data = data[1].values
# print(power_data[:10])
data = power_data.reshape(-1,1)
data = DataFrame(data)#转化维DataFrame格式
# print(data.shape)
scaler = StandardScaler()
data = scaler.fit_transform(data)
# print("标准化后的数据（前5行）:")
# print(data[:5])
# print(len(data))
#划分训练集，验证集，测试集
train_data = data[:int(len(data)*0.8)]
val_data = data[int(len(data)*0.8):int(len(data)*0.9)]
test_data = data[int(len(data)*0.9):]
# print(len(train_data),len(val_data),len(test_data))
print(train_data.shape)
#构建数据集X和Y
window_size = 96
X, Y = [], []
for i in np.arange(window_size,len(train_data)):
    X.append(train_data[i-window_size:i,:])
    Y.append(train_data[i])
x_train,y_train = np.array(X),np.array(Y)
print(x_train.shape,y_train.shape)
 

(102524, 1)
(102428, 96, 1) (102428, 1)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import wandb

def train():
    # --- 1. 初始化 WandB ---
    wandb.init(
        project="power-load-prediction", # 项目名称
        name="lstm-depth2-hidden128",    # 本次运行的名称
        config={                         # 保存超参数，方便以后对比
            "learning_rate": 0.01,
            "epochs": 20,
            "batch_size": 64,
            "hidden_size": 128,
            "num_layers": 2,
            "window_size": 96
        }
    )
    config = wandb.config

    # --- 2. 数据准备 ---
    file_path = 'load.csv'
    x_train, y_train = load_data(file_path)
    
    x_tensor = torch.from_numpy(x_train).float()
    y_tensor = torch.from_numpy(y_train).float()

    dataset = TensorDataset(x_tensor, y_tensor)
    train_loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=False)

    # --- 3. 模型、损失函数、优化器 ---
    model = LSTMModel(
        input_size=1, 
        hidden_size=config.hidden_size, 
        num_layers=config.num_layers, 
        out_size=1
    )
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

    # --- 4. 训练循环 ---
    model.train()
    for epoch in range(config.epochs):
        epoch_loss = 0.0
        for batch_x, batch_y in train_loader:
            # 清空梯度
            optimizer.zero_grad()
            
            # 前向传播
            outputs = model(batch_x)
            
            # 计算损失
            loss = criterion(outputs, batch_y.view(-1, 1))
            
            # 反向传播
            loss.backward()
            
            # 更新参数
            optimizer.step()
            
            epoch_loss += loss.item()

        # 计算平均每个 batch 的损失
        avg_loss = epoch_loss / len(train_loader)
        
        # --- 5. 将数据上传到 WandB 云端 ---
        # 在这里我们记录 Loss，回归任务中 Loss 越小代表越“准确”
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_loss,
            "learning_rate": optimizer.param_groups[0]['lr']
        })

        print(f"Epoch [{epoch+1}/{config.epochs}], Loss: {avg_loss:.6f}")

    # 训练结束，关闭 WandB 运行
    wandb.finish()

if __name__ == "__main__":
    train()